# Loisel-Privault Finite-Time Sensitivity Reproduction

Source family: Loisel and Privault integration-by-parts
representation for infimum-density / finite-time ruin
sensitivity.

Paper-specific target: compare the IBP estimate of
`d psi(u,T) / du` with a finite-difference derivative of the
exact exponential finite-time ruin formula at `u = 0.5, 1.0,
1.5`.

Reviewer rubric: please separate comments into mathematical
correctness, API/usability, and teaching value.

In [ ]:
import numpy as np

from ruin_theory import (
    CramerLundbergProcess,
    estimate_finite_time_ruin_sensitivity_ibp,
    exponential,
    finite_time_ruin_exponential,
)

model = CramerLundbergProcess(
    premium_rate=1.0,
    claim_arrival_rate=0.5,
    claim_distribution=exponential(rate=1.0),
)
surplus = np.array([0.5, 1.0, 1.5])

estimate = estimate_finite_time_ruin_sensitivity_ibp(
    model,
    surplus,
    horizon=2.0,
    n_simulations=50_000,
    seed=123,
)

step = 1e-4
finite_difference = np.array(
    [
        (
            finite_time_ruin_exponential(model, u=float(value + step), horizon=2.0)
            - finite_time_ruin_exponential(model, u=float(value - step), horizon=2.0)
        )
        / (2.0 * step)
        for value in surplus
    ]
)

comparison = np.column_stack(
    [
        surplus,
        estimate.ruin_probability_derivative,
        finite_difference,
        estimate.standard_error,
    ]
)
comparison

In [ ]:
np.testing.assert_allclose(
    estimate.ruin_probability_derivative,
    finite_difference,
    atol=4.0 * np.max(estimate.standard_error),
)
np.testing.assert_allclose(
    estimate.density,
    -estimate.ruin_probability_derivative,
)